# The Buffalo Times: News Intelligence Dashboard

**Assignment:** Extra credit, EAS 503
**Student:** Shruti Priya (50757311)

**Live app:** https://newsappeas503-ihbxry75s9tguguaepuerb.streamlit.app
**Code:** https://github.com/shruti23-ui/Newsapp_eas503

The class news app fetched headlines, ran sentiment and NER, and asked Gemini for a summary, but it used
very little of what it computed. This notebook walks through the three features built from those gaps:

| # | Feature | Missed opportunity | What it computes |
|---|---|---|---|
| 1 | Entity Explorer | spaCy entities were extracted but never used | Ranked entities, mean sentiment, entity × category, co-mentions |
| 2 | Echo Index | Each class notebook saved a different day's news, never compared | How much each headline repeats earlier days (TF-IDF cosine), rising and fading words |
| 3 | Newsroom Check | Gemini's JSON summary was never parsed or checked | How much of the news the summary covers, and which claims are weakly supported |

It runs **offline** on the four real NewsAPI snapshots saved in the class notebooks, so every number
below can be reproduced without API keys. The dashboard itself is `app.py` (run `python app.py`).

## 1. Setup

In [1]:
# Offline mode: use the saved class snapshots so the results are reproducible without API keys
import os
os.environ["DEMO_MODE"] = "true"
os.environ["ENABLE_ZSC"] = "false"   # the zero-shot model is optional (2 GB); TextBlob labels are used instead

import pandas as pd
import plotly.express as px
import plotly.io as pio

from buffalo import ai_summary, analytics, config, data_pipeline, nlp

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_colwidth", 90)

## 2. Data
Four NewsAPI `top-headlines` responses (7 categories each) saved in the class notebooks.

In [2]:
df, status = data_pipeline.get_news_data()
print(status["message"])
print(df["uid"].nunique(), "unique headlines")
df.pivot_table(index="snapshot", columns="category", values="uid", aggfunc="nunique")

Demo mode: saved class snapshots (no API key set). Add NEWS_API_KEY to .env for live news.
490 unique headlines


category,business,entertainment,general,health,science,sports,technology
snapshot,,,,,,,
2026-10-19,19,20,17,19,19,19,18
2026-10-21,20,18,19,19,19,20,20
2026-10-26,16,20,18,18,19,20,18
2026-11-02,18,19,19,18,20,20,20


In [3]:
# TextBlob polarity and subjectivity, sentiment labels, and spaCy entities for every headline
df, sentiment_method = nlp.enrich(df)
print("Sentiment labels:", sentiment_method)
df[["snapshot", "category", "title", "polarity", "subjectivity", "zsc_sentiment", "entities"]].head(6)

Sentiment labels: TextBlob fallback - zero-shot model disabled (ENABLE_ZSC=false)


,snapshot,category,title,polarity,subjectivity,zsc_sentiment,entities
0,2026-10-19,business,"Stock Market Today: Dow turns up, S&P 500 and Nasdaq rally as tech stocks bounce; oil ...",0.000000,0.000000,neutral,"[(Nasdaq, ORG)]"
1,2026-10-19,business,Oil Futures Pull Back on Russia Sanctions Waiver,0.000000,0.000000,neutral,[]
2,2026-10-19,business,Uber wins bid to throw out £200m black cab claims,0.066667,0.316667,positive,[]
3,2026-10-19,business,SpaceX surprises Wall Street with Oct. 20 earnings date — here's why,0.000000,0.000000,neutral,"[(SpaceX, ORG)]"
4,2026-10-19,business,Musk pledges 1 SpaceX share for every child in America,0.000000,0.000000,neutral,"[(SpaceX, ORG), (United States, GPE)]"
5,2026-10-19,business,CLF Stock Flashes Buy Signal On Steelmaker's Massive $4 Bil Outlook,0.000000,1.000000,neutral,"[(Flashes Buy Signal, PERSON)]"


### Data-quality note: is a "top headline" just the newest one?
Spearman correlation between NewsAPI's order and publish time, inside each snapshot and category.
A value of −1 means the list is sorted exactly newest-first.

In [4]:
from scipy.stats import spearmanr

rows = []
for (snap, cat), g in df.groupby(["snapshot", "category"]):
    g = g.dropna(subset=["publishedAt"])
    if len(g) > 2:
        api_order = list(range(len(g)))
        rho = spearmanr(api_order, g["publishedAt"].astype("int64")).statistic
        rows.append({"snapshot": snap, "category": cat, "spearman": round(rho, 2)})
order = pd.DataFrame(rows)
print(len(order), "groups | mean Spearman:", round(order["spearman"].mean(), 2),
      "| groups at -1.00:", int((order["spearman"] == -1).sum()))

28 groups | mean Spearman: -1.0 | groups at -1.00: 28


## 3. Feature 1: Entity Explorer
spaCy `en_core_web_sm` mislabels Title Case headlines (it called "AI" a country and Trump an organization).
An `EntityRuler` placed before the statistical NER fixes known names first, then entities are counted.

In [5]:
E = analytics.entity_rows(df)
labels = dict(E.drop_duplicates("entity")[["entity", "label"]].values)
{name: labels.get(name) for name in ["Donald Trump", "Nvidia", "AI"]}

{'Donald Trump': 'PERSON', 'Nvidia': 'ORG', 'AI': None}

In [6]:
top = analytics.entity_summary(E).head(15)
top.round(3)

,entity,mentions,label,mean_polarity,snapshots
292,NFL,16,ORG,0.151,4
105,Donald Trump,12,PERSON,0.061,4
435,United States,12,GPE,-0.003,4
113,Earth,7,LOC,-0.008,4
397,SpaceX,7,ORG,0.069,4
22,Anthropic,7,ORG,-0.051,3
23,Apple,6,ORG,0.058,3
464,Xbox,6,ORG,0.143,2
299,Netflix,5,ORG,0.086,4
275,Microsoft,5,ORG,-0.020,3


In [7]:
fig = px.bar(top, x="mentions", y="entity", orientation="h", color="mean_polarity",
             color_continuous_scale="RdBu", range_color=[-0.5, 0.5],
             title="Most mentioned entities (color = average headline sentiment)")
fig.update_yaxes(autorange="reversed")
fig.show()

In [8]:
# Entity x category: where each entity appears
names = list(top["entity"])
(E[E["entity"].isin(names)].drop_duplicates(["uid", "entity"])
   .pivot_table(index="entity", columns="category", values="uid", aggfunc="count", fill_value=0)
   .reindex(names))

category,business,entertainment,general,health,science,sports,technology
entity,,,,,,,
NFL,0,0,0,0,0,16,0
Donald Trump,3,2,7,0,0,0,0
United States,2,1,3,3,1,1,1
Earth,0,0,0,0,7,0,0
SpaceX,5,0,0,0,2,0,0
Anthropic,4,1,1,0,0,0,1
Apple,1,0,0,0,0,0,5
Xbox,0,0,0,0,0,0,6
Netflix,2,3,0,0,0,0,0


In [9]:
# Co-mentions: entities that appear in the same headline
pairs = analytics.cooccurrence(E, names)
pd.DataFrame([(a, b, n) for (a, b), n in pairs.most_common(8)],
             columns=["entity A", "entity B", "headlines together"])

,entity A,entity B,headlines together
0,Iran,United States,3
1,Donald Trump,Iran,2
2,Donald Trump,United States,2
3,SpaceX,United States,1
4,NFL,Patriots,1
5,Google,Microsoft,1
6,Microsoft,Xbox,1
7,Anthropic,Donald Trump,1


## 4. Feature 2: Echo Index
Each headline is turned into a TF-IDF vector. Its **echo score** is the highest cosine similarity to any
headline from an earlier snapshot: 0 means a brand new story, 1 means the same headline again.

In [10]:
echo = analytics.echo_scores(df)
trend = echo.groupby("snapshot")["echo"].mean().round(3)
trend

snapshot
2026-10-19      NaN
2026-10-21    0.118
2026-10-26    0.163
2026-11-02    0.186
Name: echo, dtype: float64

In [11]:
px.line(trend.reset_index(), x="snapshot", y="echo", markers=True,
        title="Average echo score per snapshot (higher = more repeated news)").show()

In [12]:
# Most repeated headlines and the earlier headline each one echoes
echo.sort_values("echo", ascending=False)[["snapshot", "title", "echo", "echo_of", "echo_of_snapshot"]].head(5).round(2)

,snapshot,title,echo,echo_of,echo_of_snapshot
444,2026-11-02,"Game ball, three stars, and snap count analysis: Week 8, Eagles at Commanders",0.81,"Game ball, three stars, and snap count analysis: Week 6, Eagles vs. Panthers",2026-10-19
272,2026-10-26,"Horoscope for Monday, October 26, 2026",0.74,"Horoscope for Monday, October 19, 2026",2026-10-19
451,2026-11-02,5 winners and losers from Patriots’ win over the Dolphins,0.74,6 winners and losers from Patriots’ win over Jets,2026-10-19
286,2026-10-26,How hot you drink your tea may affect heart health,0.53,How you drink your tea could determine whether you gain or lose heart health benefits,2026-10-21
320,2026-10-26,"NFL Week 7 winners, losers led by Jets' Aaron Glenn running out of chances",0.53,The Winners and Losers of the NFL Week 6,2026-10-19


In [13]:
# Rising and fading words: log-odds with an informative Dirichlet prior (Monroe, Colaresi & Quinn, 2008)
focus = df["snapshot"].max()
terms = analytics.term_shift(df, focus)
print("Largest |z|:", round(terms["z"].abs().max(), 2), "(below 2, so these trends are directional, not significant)")
pd.concat([terms.head(6), terms.tail(6)]).round(2)

Largest |z|: 1.69 (below 2, so these trends are directional, not significant)


,word,z,focus_count,earlier_count
0,jack,1.69,3,0
1,billion,1.48,5,3
2,defense,1.38,3,1
3,calls,1.38,3,1
4,got,1.38,3,1
5,midterm,1.38,3,1
269,secret,-0.91,0,6
270,oct,-0.91,0,6
271,iphone,-0.91,0,6
272,best,-0.98,0,7


## 5. Feature 3: Newsroom Check
The class app asked Gemini for a JSON summary but showed a hardcoded string. Here the real saved Gemini output
is parsed and audited: a headline counts as covered when its TF-IDF cosine similarity to a summary cluster is
at least the threshold.

In [14]:
summary = ai_summary.load_summaries()["2026-11-02"]
print(summary["global_summary"])
[c["title"] for c in summary["clusters"]]

Today's news spans dynamic financial markets reacting to midterm elections and corporate tech maneuvers, alongside critical health warnings regarding bird flu and ongoing disease outbreaks. In the sports arena, fans are processing major NFL injuries and weekend game takeaways, while international updates cover geopolitical maneuvers and conflict developments in the Middle East.


['Stock Market and Corporate Financials',
 'Sports Updates and Injuries',
 'Health and Disease Outbreaks',
 'Geopolitics and International Conflict']

In [15]:
audit = analytics.audit_summary(df[df["snapshot"] == "2026-11-02"], summary)
cov = analytics.coverage_by_category(audit["heads"], 0.10, config.CATEGORIES)
covered, total = cov["represented"].sum(), cov["total"].sum()
print(f"Headlines covered: {covered} of {total} ({covered / total:.0%})")
print("Categories left out:", list(cov.loc[cov["represented"] == 0, "category"]))
cov.round(3)

Headlines covered: 14 of 125 (11%)
Categories left out: ['entertainment', 'science', 'technology']


,category,represented,total,share
0,business,5,18,0.278
1,entertainment,0,19,0.000
2,health,3,18,0.167
3,science,0,20,0.000
4,sports,5,20,0.250
5,technology,0,20,0.000
6,general,1,10,0.100


In [16]:
px.bar(cov, x="share", y="category", orientation="h", range_x=[0, 1],
       title="Share of each category's headlines covered by the AI summary (threshold 0.10)").show()

In [17]:
# Threshold sweep: the conclusion does not depend on one cut-off
sweep = []
for th in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    c = analytics.coverage_by_category(audit["heads"], th, config.CATEGORIES)
    sweep.append({"threshold": th, "coverage": round(c["represented"].sum() / c["total"].sum(), 3),
                  "categories left out": int((c["represented"] == 0).sum())})
pd.DataFrame(sweep)

,threshold,coverage,categories left out
0,0.05,0.216,1
1,0.10,0.112,3
2,0.15,0.080,4
3,0.20,0.072,4
4,0.25,0.040,4
5,0.30,0.024,4


In [18]:
# Weakest claims: summary sentences with the least support in the headlines
audit["sents"].sort_values("support")[["where", "sentence", "support", "best_headline"]].head(5).round(2)

,where,sentence,support,best_headline
0,Geopolitics and International Conflict,"Concurrently, regional security reports highlight ongoing challenges and cyber inciden...",0.08,Washington reports first US bird flu death of 2026; health officials say risk to the p...
1,Global summary,"In the sports arena, fans are processing major NFL injuries and weekend game takeaways...",0.17,"NFL trade deadline live updates: Latest news, rumors after Week 8"
2,Health and Disease Outbreaks,"Additionally, new studies continue to shed light on various medical conditions, rangin...",0.20,"Energy drinks linked to higher risk of cardiac arrest in young adults, study finds"
3,Global summary,Today's news spans dynamic financial markets reacting to midterm elections and corpora...,0.23,2026 Midterm Elections Live Updates
4,Stock Market and Corporate Financials,"Meanwhile, several major tech and finance companies are making moves, including Anthro...",0.32,"Oracle files to sell $20 billion in new stock, bonds to finance AI buildout"


In the app, the next steps use live APIs: **Tavily** searches the web for the weakest claims, **Gemini**
rewrites the summary to cover every category (re-scored with the same audit), and **ElevenLabs** reads it aloud.
The cell below runs the rewrite when `RUN_LIVE = True` and a `GEMINI_API_KEY` is in `.env`.

In [19]:
RUN_LIVE = False

if RUN_LIVE:
    from buffalo import newsroom
    snap_df = df[df["snapshot"] == "2026-11-02"]
    bulletin, status = newsroom.rewrite_balanced("2026-11-02", snap_df)
    after = analytics.audit_summary(snap_df, newsroom.bulletin_as_summary(bulletin))["heads"]
    c = analytics.coverage_by_category(after, 0.10, config.CATEGORIES)
    print(status, "| coverage after rewrite:", round(c["represented"].sum() / c["total"].sum(), 3))
else:
    print("Skipped (offline). Set RUN_LIVE = True to call Gemini.")

Skipped (offline). Set RUN_LIVE = True to call Gemini.


## 6. Supporting analysis: how opinionated is the wording?
TextBlob subjectivity was computed in class but never shown (0 = factual, 1 = opinion).

In [20]:
arts = analytics.unique_articles(df)
(arts.groupby("category")
     .agg(headlines=("uid", "count"), mean_polarity=("polarity", "mean"), mean_subjectivity=("subjectivity", "mean"),
          share_factual=("subjectivity", lambda s: (s < 0.5).mean()))
     .sort_values("mean_subjectivity", ascending=False).round(3))

,headlines,mean_polarity,mean_subjectivity,share_factual
category,,,,
technology,76,0.079,0.388,0.645
entertainment,77,-0.036,0.327,0.688
sports,79,0.138,0.312,0.684
general,34,0.010,0.300,0.706
health,74,0.062,0.295,0.743
science,77,0.020,0.287,0.701
business,73,0.081,0.258,0.753


## 7. Reproduce
The test suite runs offline on the same data.

In [21]:
import subprocess
import sys

result = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no"], capture_output=True, text=True)
print(result.stdout.strip().splitlines()[-1])

10 passed, 4 warnings in 29.15s


**Run the dashboard:**

```bash
python -m venv .venv
.venv\Scripts\activate        # macOS/Linux: source .venv/bin/activate
pip install -r requirements.txt
python app.py                   # http://127.0.0.1:8080
```

Add `NEWS_API_KEY`, `GEMINI_API_KEY`, `TAVILY_API_KEY` and `ELEVENLABS_API_KEY` to a `.env` file for live news
(today and the past 7 days), the Newsroom Check steps and the radio bulletin.